In [ ]:
import os
import sys
from pathlib import Path
import torch

REPO_ROOT = "/vol/biomedic3/tx1215/REPA-E"
FAILURE_DIR = "/vol/biomedic3/tx1215/REPA-E/checkpoints/embed_latent_dit_sits2_256x192_debug/failures/step_0037386"
DEVICE = torch.device("cuda:0")

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from models.sit import SiT_models

def load_pt(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

ckpt = load_pt(Path(FAILURE_DIR) / "model_before_failure.pt")
rank0 = load_pt(Path(FAILURE_DIR) / "rank_0.pt")
rank1 = load_pt(Path(FAILURE_DIR) / "rank_1.pt")
args = ckpt["args"]

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("failing step:", ckpt.get("failing_step"))
print("rank0 keys:", list(rank0.keys()))
print("rank1 keys:", list(rank1.keys()))
print("rank0 bad gradients:", rank0.get("bad_grad_stats"))
print("rank1 bad gradients:", rank1.get("bad_grad_stats"))

In [ ]:
def getarg(name, default=None):
    return getattr(args, name, default)

def move_to(x):
    if torch.is_tensor(x):
        return x.to(DEVICE)
    if isinstance(x, dict):
        return {k: move_to(v) for k, v in x.items()}
    if isinstance(x, list):
        return [move_to(v) for v in x]
    if isinstance(x, tuple):
        return tuple(move_to(v) for v in x)
    return x

def stats(x):
    x = x.detach()
    finite = torch.isfinite(x)
    xf = x[finite].float()
    return {
        "shape": tuple(x.shape),
        "rms": torch.sqrt((xf ** 2).mean()).item() if xf.numel() else float("nan"),
        "std": xf.std(unbiased=False).item() if xf.numel() else float("nan"),
        "absmax": xf.abs().max().item() if xf.numel() else float("nan"),
        "nan": torch.isnan(x).sum().item(),
        "inf": torch.isinf(x).sum().item(),
    }

def restore_rng(snapshot):
    state = snapshot.get("forward_rng_state")
    if state is None:
        return
    if state.get("cpu") is not None:
        torch.set_rng_state(state["cpu"])
    if state.get("cuda") is not None:
        torch.cuda.set_rng_state(state["cuda"], device=DEVICE)

def precision_context(snapshot):
    precision = snapshot.get("mixed_precision", getarg("mixed_precision", "bf16"))
    if precision == "bf16":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    if precision == "fp16":
        return torch.autocast("cuda", dtype=torch.float16)
    return torch.autocast("cuda", enabled=False)

def infer_z_dims(state):
    z_dims = []
    i = 0
    while f"projectors.{i}.4.weight" in state:
        z_dims.append(state[f"projectors.{i}.4.weight"].shape[0])
        i += 1
    return z_dims

In [ ]:
def build_model(snapshot):
    z = snapshot["z"]
    latent_size = tuple(z.shape[-2:])
    in_channels = z.shape[1]
    state = ckpt["model"]

    model = SiT_models[getarg("model")](
        input_size=latent_size,
        in_channels=in_channels,
        num_classes=getarg("num_classes", 1000),
        class_dropout_prob=getarg("cfg_prob", 0.1),
        z_dims=infer_z_dims(state),
        encoder_depth=getarg("encoder_depth", 8),
        bn_momentum=getarg("bn_momentum", 0.1),
        fused_attn=snapshot.get("fused_attn", getarg("fused_attn", True)),
        qk_norm=snapshot.get("qk_norm", getarg("qk_norm", False)),
    )

    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.train()
    model.bn.eval()
    return model

In [ ]:
def register_hooks(model):
    trace = {}
    handles = []

    def hook_layer(name, module):
        def hook(module, inputs, output):
            x = inputs[0]
            y = output

            trace[name] = {
                "forward_in": stats(x),
                "forward_out": stats(y),
            }

            if x.requires_grad:
                x.register_hook(lambda g, n=name: trace[n].update({"grad_in": stats(g)}))

            if y.requires_grad:
                y.register_hook(lambda g, n=name: trace[n].update({"grad_out": stats(g)}))

        handles.append(module.register_forward_hook(hook))

    hook_layer("PatchEmbed", model.x_embedder)

    for i, block in enumerate(model.blocks):
        hook_layer(f"Block {i}", block)

    hook_layer("FinalLayer", model.final_layer)

    return trace, handles

In [ ]:
def run_rank(snapshot, rank_name):
    model = build_model(snapshot)
    trace, handles = register_hooks(model)

    z = move_to(snapshot["z"])
    parents = move_to(snapshot["parents"])
    zs = move_to(snapshot.get("zs"))

    time_input = move_to(snapshot.get("time_input")) if snapshot.get("time_input") is not None else None
    noises = move_to(snapshot.get("noises")) if snapshot.get("noises") is not None else None

    if time_input is None or noises is None:
        restore_rng(snapshot)

    model.zero_grad(set_to_none=True)

    with precision_context(snapshot):
        out = model(
            x=z,
            parents=parents,
            zs=zs,
            loss_kwargs={
                "weighting": getarg("weighting", "uniform"),
                "path_type": getarg("path_type", "linear"),
                "prediction": getarg("prediction", "v"),
                "align_only": False,
            },
            time_input=time_input,
            noises=noises,
        )

        denoise = out["denoising_loss"].mean()
        proj = out["proj_loss"].mean()
        loss = denoise + getarg("proj_coeff", 0.0) * proj

    loss.backward()
    for name, p in model.named_parameters():
        if p.grad is None:
            continue

        if not torch.isfinite(p.grad).all():
            g = p.grad
            finite = g[torch.isfinite(g)]

            print(
                name,
                "absmax =", finite.abs().max().item() if finite.numel() else float("nan"),
                "nan =", torch.isnan(g).sum().item(),
                "+inf =", torch.isposinf(g).sum().item(),
                "-inf =", torch.isneginf(g).sum().item(),
            )

    for h in handles:
        h.remove()

    print(f"{rank_name} loss: {loss.item():.6g}")
    print("saved loss:", snapshot.get("losses", {}).get("sit_loss"))

    return model, trace

In [ ]:
def print_trace(trace, title):
    print(f"\n===== {title} =====")
    print(
        f"{'Layer':12s} "
        f"{'In RMS':>10s} {'Out RMS':>10s} {'Fwd gain':>10s} "
        f"{'Grad out':>10s} {'Grad in':>10s} {'Bwd gain':>10s} "
        f"{'Out max':>10s} {'GIn max':>10s} {'bad':>6s}"
    )

    for name, t in trace.items():
        fi = t["forward_in"]
        fo = t["forward_out"]
        gi = t.get("grad_in")
        go = t.get("grad_out")

        fwd_gain = fo["rms"] / fi["rms"] if fi["rms"] > 0 else float("nan")
        bwd_gain = gi["rms"] / go["rms"] if gi is not None and go is not None and go["rms"] > 0 else float("nan")

        grad_out_rms = go["rms"] if go is not None else float("nan")
        grad_in_rms = gi["rms"] if gi is not None else float("nan")
        grad_in_max = gi["absmax"] if gi is not None else float("nan")

        bad = fo["nan"] + fo["inf"]
        if gi is not None:
            bad += gi["nan"] + gi["inf"]
        if go is not None:
            bad += go["nan"] + go["inf"]

        print(
            f"{name:12s} "
            f"{fi['rms']:10.4g} {fo['rms']:10.4g} {fwd_gain:10.4g} "
            f"{grad_out_rms:10.4g} {grad_in_rms:10.4g} {bwd_gain:10.4g} "
            f"{fo['absmax']:10.4g} {grad_in_max:10.4g} {bad:6d}"
        )

In [ ]:
model0, trace0 = run_rank(rank0, "Rank 0")
print_trace(trace0, "RANK 0")

In [ ]:
del model0
torch.cuda.empty_cache()

model1, trace1 = run_rank(rank1, "Rank 1")
print_trace(trace1, "RANK 1")